# Step 1. Build the two-relation patient networks (BPJS TB)

**Main network (Prof Iin, 11 Nov 2026):** one weighted patient-to-patient network:
- **household** (same `PSTV02`), weight **1.0**
- **same primary-care clinic + same symptom/diagnosis**, weight **0.5**
- a pair linked by both: **1.5**

This notebook only builds the networks and saves them. Community detection is done by one notebook per algorithm.

| Step | Notebook | Code file | Output folder (`graph_TB_ieee/results/two_relations/`) |
|---|---|---|---|
| 0 | `net_00_dataset.ipynb` | `common.py` | `dataset/` |
| 1 | `net_01_build.ipynb` | `net_common.py` | `networks/` |
| 2 | `net_02_louvain.ipynb` | `net_louvain.py` | `louvain/` |
| 3 | `net_03_leiden.ipynb` | `net_leiden.py` | `leiden/` |
| 4 | `net_04_label_propagation.ipynb` | `net_label_propagation.py` | `label_propagation/` |
| 5 | `net_05_girvan_newman.ipynb` | `net_girvan_newman.py` | `girvan_newman/` |
| 6 | `net_06_compare.ipynb` | `net_common.py` | `compare/` |
| 7 | `net_07_figures.ipynb` | – | `figs/<algorithm>/` |
| 8 | `net_08_runtime.ipynb` | – | `runtime/` |

Step 0 (dataset profile) is independent. Step 1 runs once. Steps 2–5 are independent: run them in any order, each one reads the networks from step 1. Every build and algorithm run appends its timing to `runtime/runtime_log.csv`; step 8 turns it into the runtime table.

## Node and edge definitions

### Nodes
| Item | Definition | Source column |
|---|---|---|
| Node | one patient (peserta) | `PSTV01` (Membership) |
| TB cohort | patient with ≥ 1 ICD-10 code A15–A19 in 2024, in any table | `FKP14A` (FKTP), `PNK13A` (non-cap), `FKL15A`/`FKL17A` (FKRTL), `FKL24A` (secondary dx) |
| R1 nodes | TB cohort + sampled members of their households (contacts) | `PSTV02` |
| R2 nodes | TB cohort patients with ≥ 1 FKTP visit in 2024 | `FKP03` (date) |
| Node attributes | household, sex, district of residence, TB-coded, died, sampling weight | `PSTV02`, `PSTV05`, `PSTV10`, `PSTV18`/`PSTV17`, `PSTV15` |

### Edges
| Network | Tie between patient *i* and *j* exists if … | Weight | Source columns |
|---|---|---|---|
| **R1 Household** | same family number | **1.0** | `PSTV02` |
| **R2 Clinic + diagnosis** | both have an FKTP visit at the **same clinic** (proxy: same district/city, optionally + FKTP type and ownership) with the **same 3-char ICD-10**, clinic and code from the same visit; **Z codes excluded** | **0.5** (binary: however many codes are shared) | `FKP06` (+ `FKP08`, `FKP07`), `FKP14A` |
| **R2 sensitivity** | as R2 main, but **Z20** (contact with/exposure to communicable disease) and **Z11** (screening for infectious disease) are kept | same | `FKP06`, `FKP14A`, `FKP15` |
| **R1 + R2 weighted (MAIN)** | tie in R1 or R2 | 1.0, 0.5, or 1.5 if both | |

**"Same klinik pratama":** the BPJS sample has no facility ID: FKTP visits carry only province, district, ownership, type and class (`FKP05`–`FKP09`). The clinic is therefore approximated by the district (`r2_facility_cols = ()`), or more strictly by district + FKTP type (+ ownership) (`r2_facility_cols = ("FKP08", "FKP07")`). Two patients with the same proxy may still have attended different clinics.

Edges are undirected, with no self-loops. They are built from a patient × key incidence matrix **B** (key = household for R1, `district|ICD` for R2) and **W = BBᵀ** (upper triangle). Each key therefore forms a clique of the patients who share it.

### Worked example (R2)
| Patient | FKTP visit district (`FKP06`) | ICD-10 (`FKP14A`) |
|---|---|---|
| A | 3273 (Kota Bandung) | A15, J18 |
| B | 3273 (Kota Bandung) | A15, J18, Z09 |
| C | 3171 (Jakarta Selatan) | A15 |

- A–B: shared keys `3273|A15` and `3273|J18`, so they are tied: R2 weight **0.5** (binary). Z09 is ignored. If A and B are also in the same household, their weight in the main network is 1.0 + 0.5 = **1.5**.
- A–C and B–C: same code but different district, so **no tie**.

### Why Z codes are excluded
ICD-10 chapter XXI (Z00–Z99) records the reason for contact with health services, for example Z00 check-up, Z09 follow-up after treatment and Z23 immunisation. These codes are neither a diagnosis nor a symptom; symptoms are R codes, which are kept. Common Z codes such as Z09 would also tie large numbers of unrelated patients. Z20 and Z11 can mark TB contact and screening, so they are tested in a sensitivity analysis. The TB-specific sub-codes (Z20.1, Z11.1) are not available in this dataset (`FKP15` holds the 3-character code and name), so the 3-character codes are used; in a TB-only cohort they are most likely TB-related.

### What to expect
- **R1:** households are disconnected cliques, so Louvain returns the households themselves (`louvain_vs_household_NMI` ≈ 1). This is a property of the relation, not an error.
- **R2:** communities tend to follow district, split further by shared codes.

## 0. Setup (Colab)  ← all `net_*.py` and `net_*.ipynb` files sit in `bpjs-tb-graph/pipeline/`, next to `common.py`

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/2025_doktor_ITS/COLAB/PAPER 1 IEEE SNA/bpjs-tb-graph/pipeline
%pip install -q python-igraph

Mounted at /content/drive
/content/drive/MyDrive/2025_doktor_ITS/COLAB/PAPER 1 IEEE SNA/bpjs-tb-graph/pipeline


### Config
Data are read through `common.load_tables()` (same tables and cache as scripts 01–12). If the cache is missing, `01_load_and_profile.py` is run once to build it.

In [2]:
import os
import pandas as pd
from net_common import *

cfg = Config(
    source = "common",           # read via common.load_tables()
    out_dir = None,              # None = <RES_DIR>/two_relations
    year = 2024,                 # None = all years
    cohort = "tb",               # "tb" = A15-A19 patients; "all" = every member
    r1_include_contacts = True,  # household relation: add sampled household members of TB patients
    r2_sources = ("fktp",),      # add "noncap" to include non-capitation FKTP visits
    r2_exclude_z = True,         # main analysis: drop all Z00-Z99
    r2_keep_z_full = (),         # main analysis keeps no Z code
    r2_sensitivity_keep_z = ("Z20", "Z11"),  # FKP15 has no sub-codes here
    r2_icd_filter = None,        # e.g. ("A15","A16","A17","A18","A19") to tie only on TB codes
    r2_facility_cols = (),       # clinic proxy: () = district; ("FKP08","FKP07") = + type + ownership
    w_household = 1.0,           # Prof Iin: household tie
    w_clinic = 0.5,              # Prof Iin: same clinic + same symptom/diagnosis
    r2_weight = "binary",        # "binary" = 0.5 per pair; "count" = 0.5 x shared codes
)

## 1. Build and save all networks

In [3]:
try:
    summary = build_networks(cfg)
except Exception as e:
    print(f"build failed ({type(e).__name__}: {e})")
    print("-> running 01_load_and_profile.py to build the cache, then retrying")
    get_ipython().run_line_magic("run", "01_load_and_profile.py")
    summary = build_networks(cfg)
summary.T

(runtime is logged only when run as: !python <script>.py)
[membership] 118,747 rows
[fktp] 380,429 rows
[fkrtl] 235,671 rows
[fkrtl_dx] 294,733 rows
[noncap] 17,393 rows
members 118,747 | TB-coded 10,823 | cohort 10,823
R1 household: 11,063 candidate nodes, 10,710 households
  keys 10,710 | upper bound on pairs 420
  [fktp] rows 380,429 | year 2024: 380,429 | district non-missing: 380,429 | ICD-10 valid: 247,609
  rows with patient + district + ICD: 247,609 | in TB cohort: 50,301 | after Z rule: 46,221
R2 clinic_dx: 9,763 cohort patients with clinic visits, 26,912 patient-(district,ICD) pairs
  keys 14,259 | upper bound on pairs 40,222
  [fktp] rows 380,429 | year 2024: 380,429 | district non-missing: 380,429 | ICD-10 valid: 247,609
  rows with patient + district + ICD: 247,609 | in TB cohort: 50,301  kept Z rows: {'Z11': np.int64(174), 'Z20': np.int64(78)}
 | after Z rule: 46,473
R2 clinic_dx: 9,786 cohort patients with clinic visits, 27,102 patient-(district,ICD) pairs
  keys 14,393 

network,R1_household,R2_clinic_dx,R2_sens_Z20_Z11,R1R2_weighted,R1R2_weighted_sens
candidate_nodes,11063.000000,9763.00000,9786.000000,11063.00000,11063.000000
edges,420.000000,35790.00000,35853.000000,36123.00000,36183.000000
nodes_with_edges,656.000000,8774.00000,8795.000000,9062.00000,9080.000000
isolates,10407.000000,989.00000,991.000000,2001.00000,1983.000000
components,303.000000,433.00000,432.000000,464.00000,462.000000
largest_component,5.000000,3005.00000,3012.000000,3078.00000,3084.000000
total_weight,420.000000,17895.00000,17926.500000,18315.00000,18346.500000
density,0.001955,0.00093,0.000927,0.00088,0.000878
edges_household,420.000000,NaN,NaN,333.00000,330.000000
edges_clinic_dx,NaN,35790.00000,35853.000000,35703.00000,35763.000000


## 2. Checks
Largest clinic|ICD keys (they form the largest cliques) and household sizes in the sample.

In [4]:
d = net_dir(cfg)
display(pd.read_csv(f"{d}/R2_key_sizes.csv").head(15))
pd.read_csv(f"{d}/R1_household_sizes.csv")["n_patients"].value_counts().sort_index().rename("households by sampled size")

,key,n_patients,facility,icd3,district
0,PAPUA | KOTA JAYAPURA|A15,25,PAPUA | KOTA JAYAPURA,A15,PAPUA | KOTA JAYAPURA
1,PAPUA TENGAH | NABIRE|A15,25,PAPUA TENGAH | NABIRE,A15,PAPUA TENGAH | NABIRE
2,NUSA TENGGARA TIMUR | SUMBA BARAT DAYA|A15,20,NUSA TENGGARA TIMUR | SUMBA BARAT DAYA,A15,NUSA TENGGARA TIMUR | SUMBA BARAT DAYA
3,RIAU | SIAK|A15,19,RIAU | SIAK,A15,RIAU | SIAK
4,JAWA BARAT | SUMEDANG|A16,18,JAWA BARAT | SUMEDANG,A16,JAWA BARAT | SUMEDANG
5,JAWA TIMUR | PROBOLINGGO|A16,18,JAWA TIMUR | PROBOLINGGO,A16,JAWA TIMUR | PROBOLINGGO
6,JAWA BARAT | BOGOR|A16,18,JAWA BARAT | BOGOR,A16,JAWA BARAT | BOGOR
7,JAWA TIMUR | BANGKALAN|A18,17,JAWA TIMUR | BANGKALAN,A18,JAWA TIMUR | BANGKALAN
8,JAWA BARAT | BANDUNG|A18,17,JAWA BARAT | BANDUNG,A18,JAWA BARAT | BANDUNG
9,SUMATERA BARAT | PADANG PARIAMAN|A15,17,SUMATERA BARAT | PADANG PARIAMAN,A15,SUMATERA BARAT | PADANG PARIAMAN


,households by sampled size
n_patients,
1,10407
2,266
3,28
4,5
5,4


**Next:** run any of `net_02_louvain`, `net_03_leiden`, `net_04_label_propagation`, `net_05_girvan_newman`.
Rebuild here only when the network definition changes (weights, Z rule, clinic proxy, year); then rerun the algorithm notebooks.